# Panzer — Rate limiting y proteccion contra baneos

Binance impone limites de peso (REQUEST_WEIGHT) por IP. Si los superas,
recibes un HTTP 429 y eventualmente un **ban temporal de IP** (hasta 5 minutos
o mas).

Panzer incluye un rate limiter (`BinanceFixedWindowLimiter`) que:

1. Cuenta el peso acumulado en cada ventana de 1 minuto.
2. Se sincroniza con la cabecera `X-MBX-USED-WEIGHT-1M` del servidor.
3. **Duerme automaticamente** si la siguiente peticion superaria el umbral.
4. Aplica un `safety_ratio` para no apurar el limite al 100%.

Este notebook muestra como funciona todo esto en la practica.

## 1. Limites reales de Binance

Los limites se obtienen dinamicamente desde `/exchangeInfo`. Panzer los
lee al crear el cliente.

In [1]:
from panzer import BinancePublicClient

client = BinancePublicClient(market="spot")

# Limites que cargo Panzer desde exchangeInfo
print(f"REQUEST_WEIGHT por minuto: {client.limiter.max_per_minute}")
print(f"Safety ratio:             {client.limiter.safety_ratio}")
print(f"Limite efectivo:          {int(client.limiter.max_per_minute * client.limiter.safety_ratio)}")

2026-03-06 13:16:58     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)
2026-03-06 13:16:59     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.90


REQUEST_WEIGHT por minuto: 6000
Safety ratio:             0.9
Limite efectivo:          5400


## 2. Peso de cada endpoint

No todas las peticiones cuestan lo mismo. Por ejemplo, `depth` con
`limit=5000` pesa 50, mientras que `klines` pesa solo 2.

Panzer conoce los pesos de cada endpoint y los descuenta automaticamente.

In [2]:
from panzer.exchanges.binance.weights import get_weight

# Pesos de endpoints publicos comunes
endpoints = [
    ("/api/v3/klines", None),
    ("/api/v3/trades", None),
    ("/api/v3/aggTrades", None),
    ("/api/v3/depth", {"limit": "100"}),
    ("/api/v3/depth", {"limit": "500"}),
    ("/api/v3/depth", {"limit": "5000"}),
    ("/api/v3/exchangeInfo", None),
    ("/api/v3/account", None),
    ("/api/v3/myTrades", None),
    ("/api/v3/allOrders", None),
]

print(f"{'Endpoint':<30s} {'Params':<20s} {'Peso':>5s}")
print("-" * 58)
for ep, params in endpoints:
    w = get_weight("spot", ep, params)
    label = str(params) if params else ""
    print(f"{ep:<30s} {label:<20s} {w:>5d}")

Endpoint                       Params                Peso
----------------------------------------------------------
/api/v3/klines                                          2
/api/v3/trades                                         25
/api/v3/aggTrades                                       4
/api/v3/depth                  {'limit': '100'}         5
/api/v3/depth                  {'limit': '500'}        25
/api/v3/depth                  {'limit': '5000'}      250
/api/v3/exchangeInfo                                   20
/api/v3/account                                        10
/api/v3/myTrades                                       10
/api/v3/allOrders                                      20


## 3. Inspeccionar el estado del limiter

Despues de cada peticion puedes ver cuanto peso llevas consumido
y cuanto reporta el servidor.

In [3]:
# Partimos de cero en la ventana actual
print(f"Peso usado (local):    {client.limiter.used_local}")
print(f"Peso usado (servidor): {client.limiter.last_server_used}")

Peso usado (local):    0
Peso usado (servidor): None


In [4]:
# Hacemos unas cuantas peticiones y observamos como sube el contador
for i in range(5):
    client.klines("BTCUSDT", "1h", limit=10)
    print(
        f"Peticion {i+1}: "
        f"local={client.limiter.used_local}, "
        f"servidor={client.limiter.last_server_used}"
    )

Peticion 1: local=2, servidor=2
Peticion 2: local=4, servidor=4
Peticion 3: local=6, servidor=6
Peticion 4: local=8, servidor=8
Peticion 5: local=10, servidor=10


## 4. Que pasa cuando se alcanza el limite?

El limiter **duerme automaticamente** hasta que empiece la siguiente
ventana de un minuto. Esto es transparente: tu codigo sigue ejecutandose,
solo que la siguiente peticion tarda un poco mas.

Simulemos esto con un limiter de prueba con limite muy bajo.

In [5]:
import time
from panzer.rate_limit.binance_fixed import BinanceFixedWindowLimiter

# Limiter de prueba: solo 10 de peso por minuto, safety_ratio=1.0
test_limiter = BinanceFixedWindowLimiter(max_per_minute=10, safety_ratio=1.0)

print(f"Limite: {test_limiter.max_per_minute} por minuto")
print(f"Safety ratio: {test_limiter.safety_ratio}")
print()

# Consumimos 9 de 10
for i in range(9):
    test_limiter.acquire(weight=1)
    print(f"acquire(1) #{i+1} -> used_local={test_limiter.used_local}")

print(f"\nQueda 1 de capacidad. La siguiente peticion de peso 1 pasa sin dormir:")
t0 = time.time()
test_limiter.acquire(weight=1)
print(f"acquire(1) -> used_local={test_limiter.used_local}, tardo {time.time()-t0:.3f}s")

print(f"\nAhora estamos a 10/10. La siguiente peticion DORMIRA hasta el proximo minuto.")
print("(Esto podria tardar hasta 60 segundos — cancelar si no quieres esperar)")
t0 = time.time()
test_limiter.acquire(weight=1)
print(f"acquire(1) -> used_local={test_limiter.used_local}, DURMIO {time.time()-t0:.1f}s")

2026-03-06 13:17:01  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=10, weight=1, effective_limit=10). Durmiendo 58.25 segundos.


Limite: 10 por minuto
Safety ratio: 1.0

acquire(1) #1 -> used_local=1
acquire(1) #2 -> used_local=2
acquire(1) #3 -> used_local=3
acquire(1) #4 -> used_local=4
acquire(1) #5 -> used_local=5
acquire(1) #6 -> used_local=6
acquire(1) #7 -> used_local=7
acquire(1) #8 -> used_local=8
acquire(1) #9 -> used_local=9

Queda 1 de capacidad. La siguiente peticion de peso 1 pasa sin dormir:
acquire(1) -> used_local=10, tardo 0.000s

Ahora estamos a 10/10. La siguiente peticion DORMIRA hasta el proximo minuto.
(Esto podria tardar hasta 60 segundos — cancelar si no quieres esperar)
acquire(1) -> used_local=1, DURMIO 58.3s


## 5. El efecto del safety_ratio

Con `safety_ratio=0.9` (por defecto), Panzer deja un 10% de margen.
Esto protege contra desfases de reloj entre tu maquina y Binance.

| safety_ratio | Limite real (si max=6000) | Uso maximo antes de dormir |
|:---:|:---:|:---:|
| 1.0 | 6000 | 6000 — sin margen, riesgo de ban |
| 0.9 | 5400 | Panzer duerme al llegar a 5400 |
| 0.8 | 4800 | Mas conservador |
| 0.5 | 3000 | Muy conservador |

In [6]:
# Comparar limites efectivos con distintos safety_ratio
max_weight = client.limiter.max_per_minute

for ratio in [1.0, 0.9, 0.8, 0.5]:
    effective = int(max_weight * ratio)
    print(f"safety_ratio={ratio:.1f}  ->  limite efectivo = {effective} / {max_weight}")

safety_ratio=1.0  ->  limite efectivo = 6000 / 6000
safety_ratio=0.9  ->  limite efectivo = 5400 / 6000
safety_ratio=0.8  ->  limite efectivo = 4800 / 6000
safety_ratio=0.5  ->  limite efectivo = 3000 / 6000


In [7]:
# Crear un cliente con safety_ratio diferente
conservative = BinancePublicClient(market="spot", safety_ratio=0.5)

print(f"Cliente conservador:")
print(f"  max_per_minute:  {conservative.limiter.max_per_minute}")
print(f"  safety_ratio:    {conservative.limiter.safety_ratio}")
print(f"  limite efectivo: {int(conservative.limiter.max_per_minute * conservative.limiter.safety_ratio)}")

2026-03-06 13:18:00     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)
2026-03-06 13:18:00     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.50


Cliente conservador:
  max_per_minute:  6000
  safety_ratio:    0.5
  limite efectivo: 3000


## 6. Sincronizacion con el servidor

Cada respuesta HTTP de Binance incluye la cabecera `X-MBX-USED-WEIGHT-1M`
con el peso real acumulado en el servidor. Panzer usa este valor para
**corregir** su contador local si se queda por detras.

Esto es crucial: si otras aplicaciones (bots, scripts) comparten la misma
IP, el peso del servidor sera mayor que el contador local de Panzer.
Al sincronizar, Panzer se entera y ajusta su limite.

In [8]:
# Simular que el servidor reporta un peso mas alto que el local
# (p. ej. otro bot esta consumiendo peso en la misma IP)
demo_limiter = BinanceFixedWindowLimiter(max_per_minute=6000, safety_ratio=0.9)

# Usamos 5 de peso localmente
demo_limiter.acquire(weight=5)
print(f"Antes de sync:  local={demo_limiter.used_local}, servidor={demo_limiter.last_server_used}")

# El servidor dice que ya se han usado 200 (otro bot en la misma IP)
demo_limiter.update_from_headers({"X-MBX-USED-WEIGHT-1M": "200"})
print(f"Despues de sync: local={demo_limiter.used_local}, servidor={demo_limiter.last_server_used}")
print()
print("Panzer subio su contador local a 200 para reflejar el uso real del servidor.")

Antes de sync:  local=5, servidor=None
Despues de sync: local=200, servidor=200

Panzer subio su contador local a 200 para reflejar el uso real del servidor.


## 7. Que pasa SIN rate limiter?

Sin proteccion, un bucle rapido puede agotar los 6000 de peso en
segundos. Binance responde asi:

1. **HTTP 429 (Too Many Requests):** aviso. Si sigues, viene el ban.
2. **HTTP 418 (IP banned):** ban temporal de 2-5 minutos. Tu IP no
   puede hacer NINGUNA peticion a Binance.
3. **Reincidencia:** bans mas largos, potencialmente de horas.

```python
# EJEMPLO PELIGROSO (NO ejecutar sin limiter):
import requests

for i in range(10000):
    # depth con limit=5000 pesa 50 cada una
    # 120 peticiones = 6000 de peso = LIMITE COMPLETO
    # En la peticion 121 -> HTTP 429
    # Si ignoras el 429 -> HTTP 418 (ban)
    requests.get(
        "https://api.binance.com/api/v3/depth",
        params={"symbol": "BTCUSDT", "limit": 5000}
    )
```

Con Panzer esto **no puede pasar**: el limiter duerme antes de
llegar al limite.

## 8. Ejemplo real: bucle intensivo protegido

Este bucle descarga muchos datos. Panzer gestiona el peso internamente
y dormira si se acerca al limite.

In [9]:
import time

symbols = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT"]

t0 = time.time()
for sym in symbols:
    klines = client.klines(sym, "1h", limit=100)
    depth = client.depth(sym, limit=100)
    trades = client.trades(sym, limit=100)
    print(
        f"{sym}: {len(klines)} klines, "
        f"{len(depth['bids'])} bids, "
        f"{len(trades)} trades  "
        f"| local={client.limiter.used_local} servidor={client.limiter.last_server_used}"
    )

print(f"\nTotal: {time.time()-t0:.1f}s, peso final: {client.limiter.used_local}")

BTCUSDT: 100 klines, 100 bids, 100 trades  | local=52 servidor=52
ETHUSDT: 100 klines, 100 bids, 100 trades  | local=84 servidor=84
SOLUSDT: 100 klines, 100 bids, 100 trades  | local=116 servidor=116
BNBUSDT: 100 klines, 100 bids, 100 trades  | local=148 servidor=148
XRPUSDT: 100 klines, 100 bids, 100 trades  | local=180 servidor=180

Total: 3.9s, peso final: 180


## 9. Resumen

| Concepto | Detalle |
|---|---|
| **Limite por defecto** | 6000 REQUEST_WEIGHT por minuto (puede variar) |
| **Ventana** | Fija, 1 minuto (floor(epoch/60)) |
| **safety_ratio** | 0.9 por defecto (usa hasta el 90% del limite) |
| **Sincronizacion** | Via cabecera `X-MBX-USED-WEIGHT-1M` |
| **Cuando se alcanza el limite** | Panzer duerme hasta el siguiente minuto |
| **Sin limiter** | HTTP 429 -> HTTP 418 (ban de IP) |

Panzer hace todo esto de forma transparente. No necesitas gestionar
pesos manualmente — solo usa los metodos del cliente y el limiter
se encarga del resto.